# Poll Results Visualizer - Exploratory Data Analysis

**Objective:** turn a raw, messy poll export into clean, quantified, defensible insights.

This notebook walks the full workflow end to end:

`raw export -> clean -> aggregate -> compare -> visualise -> insight`

Every step is explained *and* executed, so this file doubles as documentation.

> The dataset is **synthetic** (1,200 respondents x 5 questions) but deliberately
> dirty: duplicate rows, blank answers, mixed casing, stray whitespace and three
> different date formats - exactly what a Google Forms / SurveyMonkey export looks like.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()))

from src import config, analysis, data_cleaning, data_generator, insights, visualization

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
analysis.LOGGER, insights.LOGGER = analysis.LOGGER, insights.LOGGER
visualization.apply_matplotlib_theme()
print("Libraries loaded.")

Libraries loaded.


## 2. Load the raw export

First look at the file **as received** - we do not clean it in place, we only
inspect it, so we always know what the source data looked like.

In [2]:
raw = data_cleaning.load_raw()
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(8)

[19:11:20] INFO    Loading raw dataset -> data\raw\poll_responses_raw.csv


Raw shape: 6,180 rows x 13 columns


,respondent_id,response_date,question_id,question,category,option_selected,satisfaction_score,region,age_group,gender,device,channel,open_feedback
0,R-1000,2026-10-02,Q1,Which candidate do you support in the student ...,Election Poll,Candidate A - Aarav,NaN,East,18-24,Female,Desktop,Social Media,NaN
1,R-1001,2026-09-30,Q1,Which candidate do you support in the student ...,Election Poll,CANDIDATE B - BIANCA,NaN,West,35-44,Female,Mobile,QR Poster,wish the dashboard loaded faster on slow networks
2,R-1002,30/09/2026,Q1,Which candidate do you support in the student ...,Election Poll,Candidate A - Aarav,NaN,north,45-54,Female,Mobile,Social Media,the reports export should support pdf
3,R-1003,2026-09-22,Q1,Which candidate do you support in the student ...,Election Poll,Candidate B - Bianca,NaN,Central,25-34,Female,Mobile,Email,NaN
4,R-1004,2026-09-14,Q1,Which candidate do you support in the student ...,Election Poll,NaN,NaN,East,25-34,Female,Mobile,Social Media,NaN
5,R-1005,2026-09-26,Q1,Which candidate do you support in the student ...,Election Poll,Candidate B - Bianca,NaN,East,25-34,Male,Desktop,Email,love the collaboration features for teams
6,R-1006,2026-09-20,Q1,Which candidate do you support in the student ...,Election Poll,Candidate A - Aarav,NaN,Central,25-34,Female,Mobile,Referral,NaN
7,R-1007,2026-09-26,Q1,Which candidate do you support in the student ...,Election Poll,Candidate B - Bianca,NaN,South,45-54,Male,Mobile,Email,NaN


### 2.1 What is wrong with it?

A data analyst's first job is measuring the problem, not guessing at it.

In [3]:
answers = raw["option_selected"].astype("string")

quality_checks = {
    "Rows": len(raw),
    "Exact duplicate rows": raw.duplicated().sum(),
    "Blank answers": answers.isna().sum(),
    "Answers with stray whitespace": answers.str.len().ne(answers.str.strip().str.len()).sum(),
    "Region values needing case fix": (raw["region"] != raw["region"].str.strip().str.title()).sum(),
    "Mixed date formats": int((~raw["response_date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$")).sum()),
    "Non-numeric satisfaction rows": int(pd.to_numeric(raw["satisfaction_score"], errors="coerce").isna().sum()),
}

for name, value in quality_checks.items():
    print(f"{name:>34}: {value:,}")

                              Rows: 6,180
              Exact duplicate rows: 72
                     Blank answers: 284
     Answers with stray whitespace: 384
    Region values needing case fix: 425
                Mixed date formats: 1,250
     Non-numeric satisfaction rows: 4,935


## 3. Clean

The cleaning pipeline (src/data_cleaning.py) does, in order:

1. drop exact duplicates
2. trim whitespace on every text column
3. map case/whitespace variants back to canonical labels
4. drop blank / junk answers
5. validate that each answer belongs to its question's option set
6. parse the three date formats
7. fill missing demographics with an explicit label
8. drop duplicates **again** - rows that only looked different because of the date format

In [4]:
clean, quality = data_cleaning.clean_poll_data(raw)

print("\n--- Data quality summary ---")
for key, value in quality.items():
    if key != "columns":
        print(f"{key:>34}: {value}")

[19:11:21] INFO    Cleaned 6,180 -> 5,632 rows (91.13% retained, 163 duplicates dropped)



--- Data quality summary ---
                           rows_in: 6180
                duplicates_removed: 163
           invalid_answers_removed: 385
     unknown_question_rows_removed: 0
      out_of_scope_answers_removed: 0
                 unparseable_dates: 0
    duplicates_after_normalisation: 91
                          rows_out: 5632
                     retention_pct: 91.13
                          date_min: 2026-09-14
                          date_max: 2026-10-13
                       respondents: 1200
                         questions: 5


In [5]:
clean.head()

,respondent_id,response_date,question_id,question,category,option_selected,satisfaction_score,region,age_group,gender,device,channel,open_feedback
0,R-1004,2026-09-14,Q2,Which feature should we prioritise in the next...,Product Preference,Team Collaboration,NaN,East,25-34,Female,Mobile,Social Media,
1,R-1004,2026-09-14,Q3,How satisfied are you with the current service?,Customer Feedback,5 - Very Satisfied,5.0,East,25-34,Female,Mobile,Social Media,
2,R-1004,2026-09-14,Q4,Which communication channel do you prefer for ...,Customer Feedback,Email Newsletter,NaN,East,25-34,Female,Mobile,Social Media,
3,R-1004,2026-09-14,Q5,Which product variant would you buy?,Product Preference,Starter (Free),NaN,East,25-34,Female,Mobile,Social Media,
4,R-1020,2026-09-14,Q1,Which candidate do you support in the student ...,Election Poll,Candidate A - Aarav,NaN,Central,45-54,Female,Desktop,Email,


## 4. Aggregate: option-wise votes and share

In [6]:
for question_id in sorted(clean["question_id"].unique()):
    table = analysis.option_share_table(clean, question_id)
    print(f"\n=== {question_id} - {table.iloc[0]['question'] if 'question' in table else ''}")
    display(table[["option_selected", "votes", "share_pct"]])


=== Q1 - 


,option_selected,votes,share_pct
0,Candidate A - Aarav,378,33.78
1,Candidate B - Bianca,334,29.85
2,Candidate C - Chen,266,23.77
3,Undecided,141,12.60



=== Q2 - 


,option_selected,votes,share_pct
0,Faster Search & Filters,316,28.01
1,Dark Mode & Accessibility,302,26.77
2,Offline Sync,198,17.55
3,Team Collaboration,156,13.83
4,Lower Pricing,156,13.83



=== Q3 - 


,option_selected,votes,share_pct
0,4 - Satisfied,397,35.04
1,5 - Very Satisfied,299,26.39
2,3 - Neutral,224,19.77
3,2 - Dissatisfied,146,12.89
4,1 - Very Dissatisfied,67,5.91



=== Q4 - 


,option_selected,votes,share_pct
0,WhatsApp / Telegram,436,38.93
1,Email Newsletter,226,20.18
2,In-App Notifications,192,17.14
3,SMS,146,13.04
4,Push via Web,120,10.71



=== Q5 - 


,option_selected,votes,share_pct
0,Starter (Free),340,30.04
1,Pro Monthly,307,27.12
2,Pro Annual,213,18.82
3,Team Plan,189,16.70
4,Enterprise,83,7.33


### 4.1 The headline result, with uncertainty

A share without a margin of error is an opinion. `margin_of_error` implements
the normal approximation `1.96 * sqrt(p * (1 - p) / n)`.

In [7]:
rows = []
for question_id in sorted(clean["question_id"].unique()):
    rows.append(analysis.question_summary(clean, question_id))

summary = pd.DataFrame(rows)
summary[["question_id", "leading_option", "leading_share_pct", "runner_up_share_pct",
         "lead_margin_pct", "margin_of_error_95", "total_responses"]]

,question_id,leading_option,leading_share_pct,runner_up_share_pct,lead_margin_pct,margin_of_error_95,total_responses
0,Q1,Candidate A - Aarav,33.78,29.85,3.93,2.77,1119
1,Q2,Faster Search & Filters,28.01,26.77,1.24,2.62,1128
2,Q3,4 - Satisfied,35.04,26.39,8.65,2.78,1133
3,Q4,WhatsApp / Telegram,38.93,20.18,18.75,2.86,1120
4,Q5,Starter (Free),30.04,27.12,2.92,2.67,1132


**Reading this table:** if `lead_margin_pct` is smaller than `margin_of_error_95`,
the two leading options are statistically indistinguishable - say so in the report.

## 5. Compare: where does opinion split?

In [8]:
region_wide = analysis.crosstab_share(clean, "Q1", "region")
region_wide.style.format("{:.1f}%").background_gradient(cmap="Blues", axis=None)

option_selected,Candidate A - Aarav,Candidate B - Bianca,Candidate C - Chen,Undecided
region,,,,
Central,37.4%,23.2%,23.7%,15.8%
East,29.7%,27.8%,32.5%,9.9%
North,44.2%,26.9%,16.9%,11.9%
South,24.4%,40.3%,22.1%,13.2%
West,33.2%,28.6%,25.6%,12.6%


In [9]:
age_wide = analysis.crosstab_share(clean, "Q2", "age_group")
age_wide.style.format("{:.1f}%").background_gradient(cmap="Purples", axis=None)

option_selected,Dark Mode & Accessibility,Faster Search & Filters,Lower Pricing,Offline Sync,Team Collaboration
age_group,,,,,
18-24,30.6%,30.9%,9.5%,18.5%,10.4%
25-34,23.6%,29.1%,13.8%,16.4%,17.0%
35-44,28.4%,18.4%,16.9%,22.9%,13.4%
45-54,23.6%,25.0%,18.8%,13.2%,19.4%
55+,25.6%,38.9%,15.6%,13.3%,6.7%


## 6. Trend: did response volume change over the polling window?

In [10]:
trend = analysis.trend_table(clean)
trend.tail()

,date,responses,rolling_7d,pct_of_total
25,2026-10-09,199,185.14,3.53
26,2026-10-10,193,186.00,3.43
27,2026-10-11,197,191.71,3.50
28,2026-10-12,144,190.00,2.56
29,2026-10-13,184,190.00,3.27


In [11]:
fig = plt.figure(figsize=(11, 5))
plt.fill_between(trend["date"], trend["responses"], color=config.THEME["secondary"], alpha=0.15)
plt.plot(trend["date"], trend["responses"], marker="o", markersize=3.5,
         color=config.THEME["secondary"], label="Daily responses")
plt.plot(trend["date"], trend["rolling_7d"],
         color=config.THEME["primary"], linewidth=2.8, label="7-day rolling average")
plt.title("Response volume over the polling window")
plt.xlabel("Date"); plt.ylabel("Responses"); plt.legend(frameon=False)
plt.xticks(rotation=30)
plt.show()

C:\Users\harah\AppData\Local\Temp\ipykernel_24988\2943902980.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Free-text: what are people actually saying?

In [12]:
words = analysis.word_frequencies(clean, top_n=20)
words.style.background_gradient(cmap="Greens")

,word,frequency
0,support,441
1,dashboard,302
2,add,289
3,mode,239
4,better,232
5,love,231
6,multi,157
7,language,157
8,reports,155
9,export,155


In [13]:
visualization.plot_word_cloud(analysis.word_frequencies(clean, top_n=35))
plt.show()

[19:11:22] INFO    Chart saved -> outputs\charts\07_word_cloud.png


C:\Users\harah\AppData\Local\Temp\ipykernel_24988\3732400520.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Scale questions: satisfaction and net score

In [14]:
satisfaction = analysis.question_summary(clean, "Q3")
for key in ("mean_score", "median_score", "promoters_pct", "detractors_pct", "net_promoter_score"):
    print(f"{key:>18}: {satisfaction[key]}")

        mean_score: 3.63
      median_score: 4.0
     promoters_pct: 61.43
    detractors_pct: 18.8
net_promoter_score: 42.63


## 9. Channel performance

Not every channel is equal: some collect volume, others collect *happier* answers.

In [15]:
channels = analysis.channel_performance(clean)
channels

,channel,responses,unique_respondents,avg_satisfaction,response_share_pct
0,Email,1588,337,3.74,28.20
1,Social Media,1349,287,3.68,23.95
2,Campus Outreach,1237,263,3.53,21.96
3,QR Poster,853,181,3.52,15.15
4,Referral,605,132,3.59,10.74


## 10. Key takeaways

In [16]:
bundle = analysis.full_analysis(clean)
report = insights.build_insights(clean, bundle)

leader = max(bundle["question_summaries"], key=lambda s: s["leading_share_pct"])
closest = min(bundle["question_summaries"], key=lambda s: s["lead_margin_pct"])

print("KEY TAKEAWAYS")
print("-" * 60)
print(f"1. {leader['leading_option']} leads with {leader['leading_share_pct']:.1f}% "
      f"(+/-{leader['margin_of_error_95']} pts).")
print(f"2. {closest['question_id']} is the closest race: only {closest['lead_margin_pct']:.1f} pts "
      f"separate 1st and 2nd - treat it as a statistical tie.")
print(f"3. Average satisfaction is {bundle['overall']['avg_satisfaction']}/5.")
region_wide = bundle["crosstabs"]["Q1_by_region"]
print(f"4. Regional swing for the leading option: "
      f"{region_wide.iloc[:, 0].max() - region_wide.iloc[:, 0].min():.1f} points.")
print("5. Top feedback themes: " +
      ", ".join(f"{r['word']} ({int(r['frequency'])})" for _, r in words.head(5).iterrows()))

[19:11:22] INFO    Analysed 5,632 responses across 5 questions (1,200 respondents)


KEY TAKEAWAYS
------------------------------------------------------------
1. WhatsApp / Telegram leads with 38.9% (+/-2.86 pts).
2. Q2 is the closest race: only 1.2 pts separate 1st and 2nd - treat it as a statistical tie.
3. Average satisfaction is 3.63/5.
4. Regional swing for the leading option: 19.8 points.
5. Top feedback themes: support (441), dashboard (302), add (289), mode (239), better (232)


## 11. Export

Everything the dashboard shows is reproducible from the CLI:

```bash
python main.py all       # clean -> analyse -> 10 charts -> insight report
python main.py dashboard # interactive Streamlit dashboard
```